In [ ]:
# ▶ Colab setup — run this cell first. (On BinderHub it does nothing.)
import sys, os
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/gen-hep-notebooks'):
        !git clone -b deploy --depth 1 https://github.com/livaage/gen-hep-notebooks.git /content/gen-hep-notebooks
        !pip install -q -r /content/gen-hep-notebooks/requirements-colab.txt
    sys.path.insert(0, '/content/gen-hep-notebooks')
    os.chdir('/content/gen-hep-notebooks/notebooks')

<div style="border-left: 4px solid #2563eb; background: #f8fafc; padding: 14px 18px; margin: 6px 0; border-radius: 4px; color: #1f2937;"><div style="font-weight: 600; color: #1e3a8a; margin-bottom: 8px; font-size: 1.05em;">👋 Welcome — quick reference for Cadence</div><ul style="margin: 0; padding-left: 22px; line-height: 1.65; color: #1f2937;"><li><strong>Submit an answer:</strong> each exercise cell ends with <code>check("id", answer)</code> — replace the placeholder with your answer and run the cell.</li><li><strong>Submit a plot:</strong> <code>submit_image("id", fig)</code> ships a matplotlib or Plotly figure to your teacher's dashboard.</li><li><strong>Free-text / reflections:</strong> write your response, then <code>mark_done("id")</code> to mark it complete.</li><li><strong>Stuck?</strong> <code>show_hint("id")</code> for the teacher's hint, or <code>show_solution("id")</code> after a few wrong attempts if your teacher enabled solution reveals.</li><li><strong>Your data:</strong> <code>%cadence_export_my_data</code> to download what's stored about you, <code>%cadence_delete_my_data --yes</code> to wipe it.</li></ul></div>

In [ ]:
%load_ext cadence
%cadence_session copper-crimson-67 "your name"
from cadence import check, show_hint, show_solution, mark_done, submit_image

# 01 · Variational Autoencoders

**Generative Modelling for HEP** — notebook 1 of 6.

We start from **Probabilistic PCA** — the linear, closed-form ancestor of the
VAE — to see the latent-variable skeleton in its simplest form, then make the
encoder and decoder neural to get the VAE proper.

A VAE learns a smooth, low-dimensional *latent space*: an encoder maps data to a
distribution over latent codes, a decoder maps codes back to data, and a KL term
pulls the codes toward a standard Gaussian so the space is continuous and we can
*sample* and *interpolate* in it.

We'll build the intuition on **AFHQ animal faces** (latent arithmetic — a
"cat → dog" vector), then carry the same idea to **JetNet jets**: encode jets
with a permutation-invariant GNN and walk from **quark** to **gluon** in the
latent space, tracking our recurring **jet-mass** plot.

> You complete the cells marked **Exercise** — most ask for a number or a
> line of code, a few ask for a short **written reflection**. Everything else runs as-is.

## Setup

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

import sys; sys.path.insert(0, "..")
from src.seeds import set_seed
from src.train import get_device, to_loader, train
from src.data import load_afhq_local, load_jetnet
from src.gnn import DeepSetsEncoder, DeepSetsDecoder
from src.jets import JetStandardizer
from src.jetmass import jet_mass, plot_jet_mass, jet_mass_w1, plot_jets, plot_jet_overlay

SEED = 0
LATENT_DIM = 64      # more latent capacity -> sharper reconstructions
EPOCHS = 15          # small default; on a CPU drop to ~5, on GPU push to 30+
BATCH = 128
set_seed(SEED)
device = get_device()
print(f"device={device}  latent_dim={LATENT_DIM}  epochs={EPOCHS}")

## Part A · PCA and Probabilistic PCA

Before the neural VAE, meet its **linear ancestor**. Both answer the same
question — *can we explain high-dimensional data with a few latent numbers?* —
but with different machinery.

- **PCA** finds the linear subspace that captures the most variance. It's a
  rotation + truncation: excellent for compression, but it is *not* a generative
  model — there's no probability distribution to draw new images from.
- **Probabilistic PCA (PPCA)** wraps that same subspace in a proper
  latent-variable model: a Gaussian prior on the code, a *linear* decoder, and
  Gaussian noise. That one change lets us **sample**, score likelihoods, and
  infer a posterior over codes — exactly the structure the VAE will make
  *nonlinear* in Part B.

We build intuition on **AFHQ animal faces** at low resolution.

In [ ]:
# Real AFHQ animal faces, loaded from the local data/ folder (data/afhq_32.npz,
# built once by tools/prepare_data.py). Falls back to synthetic blobs if absent.
images, labels = load_afhq_local(n=4000, image_size=32)   # (N, 3, 32, 32) in [0,1]
X = torch.as_tensor(images, dtype=torch.float32)
img_loader = to_loader(X, batch_size=BATCH)
IMG_SHAPE = X.shape[1:]

cat_idx = np.where(labels == 0)[0]   # 0 = cat
dog_idx = np.where(labels == 1)[0]   # 1 = dog
print("images:", tuple(X.shape), "| cats:", len(cat_idx), "dogs:", len(dog_idx))

In [ ]:
from sklearn.decomposition import PCA

# Flatten each image to a 3072-vector and fit PCA with LATENT_DIM components —
# the same latent budget we'll later give the VAE. PCA gives us the principal
# directions AND, for free, every piece we need to assemble PPCA below.
X_flat = X.reshape(len(X), -1).numpy()                 # (N, 3*32*32)
pca = PCA(n_components=LATENT_DIM).fit(X_flat)
print("PCA fit:", X_flat.shape[1], "pixels ->", LATENT_DIM, "components")

<!-- cadence:task ex1-pca-variance -->
### Exercise 1 — PCA: how linear is the data?

`pca.explained_variance_ratio_` gives, for each principal component, the fraction
of the *total* pixel variance that component captures. Across **all** 3072
components these fractions would add up to exactly 1. We kept only the top
`LATENT_DIM` of them, so summing *those* fractions tells us how much of the data
those few directions already explain — i.e. how close the images sit to a flat,
linear subspace.

Sum the ratios over our `LATENT_DIM` components and return it as
`variance_explained` (a number between 0 and 1). Close to 1 → the data is nearly
linear and PCA alone is a decent codec; well below 1 → there's nonlinear
structure left over for a VAE to capture.

In [ ]:
# pca.explained_variance_ratio_ holds the fraction of variance per component.
variance_explained = ...        # sum it over the LATENT_DIM components

print(f"top-{LATENT_DIM} PCA components explain {variance_explained:.1%} of pixel variance")

check("ex1-pca-variance", variance_explained)

### Visualising PCA

Three views make PCA concrete:

1. **The data in 3D PCA space** — we can only draw three axes, so we scatter the
   first **3 of the 64** components, coloured by class. Even these first three
   already pull cats / dogs / wild apart along linear directions.
2. **The component directions as images** — each principal component *is* a
   picture in pixel space (an "eigen-animal"); the data is built by adding these
   up.
3. **Progressive reconstruction** of one image — from PC1 alone, PC2 alone,
   PC1+PC2, up to all `LATENT_DIM` components. Each component adds a bit more
   structure; the linear baseline the VAE improves on.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the 3d projection)
scores = pca.transform(X_flat)                            # (N, LATENT_DIM)

# (1) data in 3D PCA space
fig = plt.figure(figsize=(5, 4))
ax = fig.add_subplot(111, projection="3d")
for cls, name in [(0, "cat"), (1, "dog"), (2, "wild")]:
    s = labels == cls
    ax.scatter(scores[s, 0], scores[s, 1], scores[s, 2], s=3, alpha=0.3, label=name)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_zlabel("PC3")
ax.set_title("data in 3D PCA space"); ax.legend(fontsize=8)
plt.show()


def as_img(vec):
    v = vec.reshape(*IMG_SHAPE).transpose(1, 2, 0)
    return ((v - v.min()) / (np.ptp(v) + 1e-8))             # normalise for display

# (2) the first 6 component directions, as images
fig, axes = plt.subplots(1, 6, figsize=(11, 2))
for k, ax in enumerate(axes):
    ax.imshow(as_img(pca.components_[k])); ax.axis("off")
    ax.set_title(f"PC{k + 1}", fontsize=9)
fig.suptitle("principal components as images (the 'directions')", fontsize=10)
plt.show()

# (3) reconstruct ONE image from PC1 alone, PC2 alone, PC1+2, then all components
i = 0
def recon_from(ks):
    z = np.zeros_like(scores[i]); z[ks] = scores[i, ks]
    return (pca.mean_ + z @ pca.components_).reshape(*IMG_SHAPE).transpose(1, 2, 0)

panels = [("original", X_flat[i].reshape(*IMG_SHAPE).transpose(1, 2, 0)),
          ("PC1", recon_from([0])), ("PC2", recon_from([1])),
          ("PC1+2", recon_from([0, 1])),
          (f"all {LATENT_DIM}", pca.inverse_transform(scores[i:i+1])[0]
           .reshape(*IMG_SHAPE).transpose(1, 2, 0))]
fig, axes = plt.subplots(1, 5, figsize=(11, 2.4))
for ax, (name, img) in zip(axes, panels):
    ax.imshow(np.clip(img, 0, 1)); ax.axis("off"); ax.set_title(name, fontsize=9)
fig.suptitle("progressive PCA reconstruction", fontsize=10)
plt.show()

### From PCA to Probabilistic PCA

PCA gave us directions but no way to *generate*. **PPCA** (Tipping & Bishop, 1999)
adds just enough to fix that: a random code, a linear decoder, and a bit of noise.

$$z \sim \mathcal N(0, I), \qquad x = W z + \mu + \varepsilon, \qquad
  \varepsilon \sim \mathcal N(0,\, \sigma^2 I)$$

In words: draw a code `z` of `LATENT_DIM` random numbers, multiply by a matrix
`W`, add the mean image `μ`, and sprinkle on a little Gaussian noise `σ²`. That's
a complete generative model — and it is *exactly* a VAE with the neural
encoder/decoder replaced by straight lines.

The neat part: we don't fit anything new. The best-fit PPCA is built directly
from the PCA we already have. Each symbol is just one attribute of `pca`:

| symbol | meaning | in code |
|---|---|---|
| $\mu$ | mean image | `pca.mean_` |
| $U$ | eigenvectors (the PC directions) | `pca.components_.T` |
| $\lambda$ | eigenvalues (variance carried by each PC) | `pca.explained_variance_` |
| $\sigma^2$ | leftover "noise" variance | `pca.noise_variance_` |

and the loading matrix is $W = U\sqrt{\lambda - \sigma^2}$ — each direction scaled
by how much variance it carries. You'll assemble it in the next cell.

<!-- cadence:task ex2-ppca-sample -->
### Exercise 2 — PPCA: a linear model you can *sample*

Turn the PCA fit into a generator, in three short lines (each spelled out in the
cell):

1. build the loading matrix `W`,
2. draw random codes `z`,
3. decode them into images `x` with the linear rule $x = W z + \mu$.

The payoff is visual: PPCA **samples** brand-new (blurry) animal faces — something
plain PCA cannot do. And because PPCA is a real probability model, it can *score*
data too: return its average log-likelihood **per pixel** as `avg_log_lik`.

In [ ]:
# PPCA reads every piece straight off the fitted `pca` object:
mu     = pca.mean_                 # μ  : the average image,           shape (D,)
sigma2 = pca.noise_variance_       # σ² : the leftover "noise" variance, a scalar
lam    = pca.explained_variance_   # λ  : the top-LATENT_DIM eigenvalues, shape (L,)
U      = pca.components_.T          # U  : eigenvectors as COLUMNS,      shape (D, L)

# 1) build the loading matrix W (its formula is in the "From PCA to Probabilistic
#    PCA" section just above) — result shape is (D, LATENT_DIM):
W = ...

# 2) draw 8 codes and decode them with  x = z @ W.T + μ   (this is x = W z + μ):
z = np.random.randn(8, LATENT_DIM)   # 8 codes, each LATENT_DIM random numbers
x = ...            # shape (8, D) -- one generated image per row

# 3) PPCA is a real probability model, so it can score data. Report the average
#    log-likelihood PER PIXEL (pca.score gives it per image; divide by n pixels):
avg_log_lik = ...  # pca.score(X_flat) / X_flat.shape[1]

check("ex2-ppca-sample", avg_log_lik)

In [ ]:
# The 8 faces PPCA just generated (from your x), plus its score:
fig, axes = plt.subplots(1, 8, figsize=(13, 2))
for ax, xi in zip(axes, x):
    ax.imshow(np.clip(xi.reshape(*IMG_SHAPE).transpose(1, 2, 0), 0, 1)); ax.axis("off")
fig.suptitle("PPCA samples:   z ~ N(0, I)   ->   x = W z + μ", fontsize=10)
plt.show()
print(f"σ² (leftover variance): {sigma2:.4f}    avg log-likelihood / pixel: {avg_log_lik:.3f}")

<!-- cadence:task ex3-ppca-blur -->
### Exercise 3 — Why does PPCA stay blurry? (reflection)

We regenerate faces two ways: keeping only the top **4** components, then the full
**64**. Run the cell and look at both rows, then write a short answer (2–3
sentences) and mark it done (there's no single right answer here):

- Why do the 4-component samples look like a bland, averaged face?
- Even with all 64 components the faces stay soft. What is it about a *linear*
  decoder $x = Wz + \mu$ that a neural decoder will be able to beat?

In [ ]:
# Given (carried over from the teacher):
# Regenerate PPCA faces at two latent sizes, so we can watch sharpness change.
def ppca_samples(n_components, n=6):
    p = PCA(n_components=n_components).fit(X_flat)
    W = p.components_.T * np.sqrt(p.explained_variance_ - p.noise_variance_)
    z = np.random.randn(n, n_components)
    return z @ W.T + p.mean_          # x = W z + μ

for L in [4, 64]:
    imgs = ppca_samples(L)
    fig, axes = plt.subplots(1, 6, figsize=(11, 1.8))
    for ax, xi in zip(axes, imgs):
        ax.imshow(np.clip(xi.reshape(*IMG_SHAPE).transpose(1, 2, 0), 0, 1)); ax.axis("off")
    fig.suptitle(f"PPCA samples with {L} components", fontsize=10)
    plt.show()

# Your reflection (2-3 sentences), replace the string below with your 2-3 sentences, then run the cell:
#   * why are the 4-component faces so bland?
#   * what can a linear decoder x = W z + mu never do that a neural one can?

check("ex3-ppca-blur", "...your 2-3 sentences here...")

## Part B · The VAE — a nonlinear PPCA

PPCA was a *linear* latent-variable model with a closed-form fit. The VAE keeps
the very same skeleton — Gaussian prior on the code, a decoder from code to data,
reconstruction + a pull toward the prior — but replaces the linear maps with
**neural networks** and, since there is no longer a closed form, *learns* an
encoder that approximates the posterior (amortised variational inference). That
nonlinearity is what lets it curve around the data instead of slicing it with a
flat subspace.

We build a small convolutional VAE on the same AFHQ faces.

In [ ]:
class ConvVAE(nn.Module):
    def __init__(self, latent=LATENT_DIM):
        super().__init__()
        c = IMG_SHAPE[0]
        def down(i, o):   # halve spatial size
            return nn.Sequential(nn.Conv2d(i, o, 4, 2, 1), nn.BatchNorm2d(o), nn.GELU())

        def up(i, o):     # double spatial size
            return nn.Sequential(nn.ConvTranspose2d(i, o, 4, 2, 1), nn.BatchNorm2d(o), nn.GELU())

        # 32 -> 16 -> 8 -> 4, channels 64/128/256 (deeper than before = sharper)
        self.enc = nn.Sequential(down(c, 64), down(64, 128), down(128, 256), nn.Flatten())
        self.fc_mu = nn.Linear(256 * 4 * 4, latent)
        self.fc_logvar = nn.Linear(256 * 4 * 4, latent)
        self.fc_dec = nn.Linear(latent, 256 * 4 * 4)
        self.dec = nn.Sequential(
            nn.Unflatten(1, (256, 4, 4)),
            up(256, 128), up(128, 64),                      # 4 -> 8 -> 16
            nn.ConvTranspose2d(64, c, 4, 2, 1), nn.Sigmoid())   # 16 -> 32

    def encode(self, x):
        h = self.enc(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparam(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + std * torch.randn_like(std)

    def decode(self, z):
        return self.dec(self.fc_dec(z))

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparam(mu, logvar)
        return self.decode(z), mu, logvar

vae = ConvVAE().to(device)
print(vae.__class__.__name__, "params:", sum(p.numel() for p in vae.parameters()))

<!-- cadence:task ex4-elbo -->
### Exercise 4 — The ELBO loss

A VAE maximises the Evidence Lower BOund. As a loss to *minimise*:

$$\mathcal{L} = \underbrace{\text{BCE}(\hat x, x)}_{\text{reconstruction}}
   + \underbrace{-\tfrac12 \sum (1 + \log\sigma^2 - \mu^2 - \sigma^2)}_{\text{KL}(q\,\|\,\mathcal N(0,1))}$$

Implement `vae_loss`, train for a few epochs, and report the **final epoch's
mean loss** as `final_loss`.

In [ ]:
def vae_loss(model, batch):
    x = batch[0]
    x_hat, mu, logvar = model(x)
    # reconstruction term (binary cross-entropy, summed over every pixel) — copy this:
    recon = ...   # F.binary_cross_entropy(x_hat, x, reduction="sum")
    # KL term (pulls the code toward a standard Gaussian) — copy this:
    kl = ...      # -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return (recon + kl) / x.shape[0]      # average over the batch

# train the VAE with your loss, then report the final-epoch mean loss:
history = train(vae, img_loader, vae_loss, epochs=EPOCHS, lr=1e-3, device=device)
final_loss = round(float(history[-1]), 3)
print("final ELBO loss:", final_loss)

check("ex4-elbo", final_loss)

<!-- cadence:task ex5-beta-vae -->
### Exercise 5 — Find the β with the best reconstruction

The VAE loss is `recon + β·KL`. **β** sets how hard we pull the codes toward a
Gaussian, and it trades off against reconstruction quality. The cell trains a
fresh VAE at a few β values and prints each one's reconstruction MSE (a few small
VAEs, ~a minute each).

Read the printed MSEs, **find the β that gives the lowest reconstruction MSE**,
and set `best_beta` to it. Notice which way it goes — and what that choice costs
you in the latent space.

In [ ]:
# Given (carried over from the teacher):
# Train a fresh VAE at several KL weights β and compare reconstruction MSE.
def train_beta_vae(beta, epochs=6):
    m = ConvVAE().to(device)
    def loss(model, batch):
        x = batch[0]; x_hat, mu, logvar = model(x)
        recon = F.binary_cross_entropy(x_hat, x, reduction="sum")
        kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        return (recon + beta * kl) / x.shape[0]
    train(m, img_loader, loss, epochs=epochs, lr=1e-3, device=device)
    m.eval()
    with torch.no_grad():
        xb = X[:512].to(device)
        return m, float(F.mse_loss(m.decode(m.encode(xb)[0]), xb))

# sweep a few β values and record each one's reconstruction MSE
betas = [0.1, 1.0, 5.0]
models, mses = [], []
for b in betas:
    m, mse = train_beta_vae(b)
    models.append(m); mses.append(mse)
    print(f"β={b}: reconstruction MSE {mse:.4f}")

# see it: the same 6 faces reconstructed at each β
with torch.no_grad():
    xb = X[:6].to(device)
    recons = [m.decode(m.encode(xb)[0]).cpu() for m in models]
rows = [("original", xb.cpu())] + [(f"β={b}", r) for b, r in zip(betas, recons)]
fig, axes = plt.subplots(len(rows), 6, figsize=(11, 2 * len(rows)))
for i, (label, imgs) in enumerate(rows):
    for j in range(6):
        axes[i, j].imshow(imgs[j].permute(1, 2, 0).clip(0, 1)); axes[i, j].axis("off")
    axes[i, 0].set_title(label, fontsize=9, loc="left")
plt.show()

# Look at the MSEs printed above. Find the β that gave the LOWEST reconstruction
# MSE and set best_beta to it:
best_beta = ...     # one of the values in `betas`

check("ex5-beta-vae", best_beta)

<!-- cadence:task ex6-latent-arithmetic -->
### Exercise 6 — Latent arithmetic: a "cat → dog" vector  🐱→🐶

The fun bit. Average the latent codes of cats and of dogs, take the difference
to get a **direction**, and add it to a cat's code. Decode along the way to make
a morph grid (we plot it for you). Then quantify the shift: the cosine
similarity between the cat→dog direction and the actual latent displacement you
applied. Return it as `direction_cos` — it should be ~1.0 if you built the
vector correctly.

In [ ]:
vae.eval()
with torch.no_grad():
    cat_mu = vae.encode(X[cat_idx][:256].to(device))[0].mean(0)   # average cat code
    dog_mu = vae.encode(X[dog_idx][:256].to(device))[0].mean(0)   # average dog code
    direction = ...        # a vector pointing from the average cat code to the average dog code
    # morph one cat toward dog in 6 steps (decoded + plotted in the next cell):
    base = vae.encode(X[cat_idx[0]][None].to(device))[0]
    steps = [base + a * direction for a in np.linspace(0, 1, 6)]
    # how well the applied shift lines up with `direction` (should be ~1.0):
    applied = (steps[-1] - steps[0]).flatten()
    direction_cos = ...    # float(F.cosine_similarity(direction.flatten(), applied, dim=0))

check("ex6-latent-arithmetic", direction_cos)

In [ ]:
# your cat -> dog morph (decode each step of the walk):
with torch.no_grad():
    grid = torch.cat([vae.decode(z) for z in steps])
fig, axes = plt.subplots(1, 6, figsize=(12, 2))
for ax, im in zip(axes, grid.cpu()):
    ax.imshow(im.permute(1, 2, 0).clip(0, 1).numpy()); ax.axis("off")
fig.suptitle("cat → dog latent morph"); plt.show()

<!-- cadence:task ex7-pca-vs-vae -->
### Exercise 7 — PCA vs. the VAE at the same budget (reflection)

Both codecs squeeze an image into `LATENT_DIM` numbers. The cell reconstructs the
same batch through **PCA** and through the **VAE** and prints both errors. Run it —
you will probably find that **PCA actually wins on raw MSE**. That is not a bug:
PCA is the *optimal* linear reconstructor for squared error, while the VAE
deliberately trades some reconstruction accuracy for a smooth latent space it can
**sample and interpolate**. Write 2–3 sentences and mark it done:

- Which reconstructed better here, and why is that the expected result?
- If PCA reconstructs at least as well, what does the VAE give you that PCA
  cannot — and when would you still reach for it?

In [ ]:
# Given (carried over from the teacher):
# Same LATENT_DIM budget, two codecs: linear PCA vs the nonlinear VAE.
batch = X_flat[:512]
pca_mse = float(np.mean((pca.inverse_transform(pca.transform(batch)) - batch) ** 2))
vae.eval()
with torch.no_grad():
    xb = X[:512].to(device)
    vae_mse = float(F.mse_loss(vae.decode(vae.encode(xb)[0]).cpu(), X[:512]))
print(f"{X_flat.shape[1] // LATENT_DIM}x compression   |   "
      f"PCA MSE {pca_mse:.4f}   vs   VAE MSE {vae_mse:.4f}")

# Your reflection (2-3 sentences), replace the string below with your 2-3 sentences, then run the cell:
#   * which codec had the LOWER reconstruction MSE, and why is that expected?
#   * PCA reconstructs fine here — so what do we actually gain from the VAE?

check("ex7-pca-vs-vae", "...your 2-3 sentences here...")

## Animate it — latent walks as gifs

Several of these ideas are more fun *moving*. `src/anim.py` has a tiny helper:

```python
from src.anim import frames_to_gif, show_gif
frames = [...]   # list of (C,H,W) tensors, HxWxC arrays, OR matplotlib figures
frames_to_gif(frames, "out.gif", fps=12, scale=6)   # scale upsizes small images
show_gif("out.gif")                                  # display inline
```

It takes model image tensors directly, and even matplotlib figures (via
`fig_to_frame`) so you can animate 2D plots — reuse it for the **denoising
trajectory** in notebook 03 or **GAN samples over epochs** in 02. Here we
animate the cat → dog latent walk:

In [ ]:
from src.anim import frames_to_gif, show_gif

vae.eval()
with torch.no_grad():
    cmu = vae.encode(X[cat_idx[:256]].to(device))[0].mean(0)
    dmu = vae.encode(X[dog_idx[:256]].to(device))[0].mean(0)
    walk = [cmu + a * (dmu - cmu) for a in np.linspace(0, 1, 24)]   # interpolate
    frames = [vae.decode(z[None])[0].cpu() for z in walk]           # 24 x (3,32,32)

gif_path = frames_to_gif(frames, "cat_to_dog.gif", fps=12, scale=6)
print("wrote", gif_path)
show_gif(gif_path)

### Why still a little soft?

Even with more capacity, VAE samples stay slightly blurry — and that's
fundamental, not a bug. The Gaussian likelihood + the KL term reward the decoder
for hedging (predicting the *average* of plausible pixels) rather than committing
to sharp detail. Levers that help, in order of impact:

- **More latent capacity / depth / epochs** (what we just did).
- **β-VAE**: down-weight the KL (`recon + β·KL`, β < 1) — sharper images, but a
  less regular latent space, so latent arithmetic gets noisier. Try it.
- **A perceptual or adversarial reconstruction loss** (e.g. VAE-GAN) — which is
  exactly the bridge to notebook **02 (GANs)** and **03 (diffusion)**, where
  we'll get crisp samples and then bring that power to jets.

So treat this softness as the baseline to beat across the next two notebooks.

## Part C · Jets in latent space

Now the physics. Jets are **point clouds** of particles — unordered sets — so we
encode them with a permutation-invariant **GNN** (DeepSets): a per-particle MLP,
a masked pool, then a head to the latent space. We train a jet VAE, then walk
the latent space from the **quark** region to the **gluon** region and watch the
jet mass.

In [ ]:
quark = load_jetnet("q", num_particles=30, max_jets=5000)   # physical (etarel,phirel,ptrel)
gluon = load_jetnet("g", num_particles=30, max_jets=5000)
jets = np.concatenate([quark, gluon]).astype(np.float32)
is_gluon = np.concatenate([np.zeros(len(quark)), np.ones(len(gluon))])

# Standardize jets (log-pt + z-score) so the model targets the right per-feature
# scales and can learn the pt concentration. We train in standardized space and
# inverse-transform samples back to physical jets for the mass plot / displays.
jet_std = JetStandardizer().fit(jets)
jets_s = jet_std.transform(jets)
jet_loader = to_loader(jets_s, batch_size=BATCH)
print("jets:", jets.shape, "| standardized for training")


class JetVAE(nn.Module):
    def __init__(self, latent=LATENT_DIM, n_particles=30):
        super().__init__()
        self.encoder = DeepSetsEncoder(in_features=3, hidden=128, latent=latent * 2,
                                       mask_padding=False)   # standardized jets
        self.decoder = DeepSetsDecoder(latent=latent, n_particles=n_particles)
        self.latent = latent

    def forward(self, x):
        h = self.encoder(x)
        mu, logvar = h[:, :self.latent], h[:, self.latent:]
        z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        return self.decoder(z), mu, logvar

jvae = JetVAE().to(device)


def jet_recon(model, batch):
    x = batch[0]
    x_hat, mu, logvar = model(x)
    recon = F.mse_loss(x_hat, x)                       # simple per-feature MSE
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + 1e-3 * kl


# Train the jet VAE up front so the exercise can focus on the latent walk.
_jhist = train(jvae, jet_loader, jet_recon, epochs=EPOCHS, lr=1e-3, device=device)
print("JetVAE params:", sum(p.numel() for p in jvae.parameters()),
      "| final recon+KL:", round(float(_jhist[-1]), 4))

### A look at the jets

Before generating any, see what a jet *is*: each particle is a dot in the
(η, φ) plane with size ∝ pₜ. Quark jets are narrow and collimated; gluon jets
are wider and busier — the structure the GNN has to learn (and the generators in
01/02 have to reproduce).

In [ ]:
plot_jets(quark[:3], titles=[f"quark {i}" for i in range(3)]); plt.show()
plot_jets(gluon[:3], titles=[f"gluon {i}" for i in range(3)]); plt.show()

### The quark → gluon latent walk

A quick look at the latent *structure*: take the mean quark code and the mean
gluon code, and decode a few points along the line between them. The jets morph
from narrow (quark-like) to wide (gluon-like) — the latent space is organised by
the physics.

In [ ]:
jvae.eval()
with torch.no_grad():
    q_s = torch.as_tensor(jet_std.transform(quark[:512])).to(device)
    g_s = torch.as_tensor(jet_std.transform(gluon[:512])).to(device)
    z_q = jvae.encoder(q_s)[:, :LATENT_DIM].mean(0)
    z_g = jvae.encoder(g_s)[:, :LATENT_DIM].mean(0)
    steps = torch.stack([(1 - a) * z_q + a * z_g
                         for a in torch.linspace(0, 1, 3, device=device)])
    walk_jets = jet_std.inverse_transform(jvae.decoder(steps).cpu().numpy())  # -> physical
plot_jets(walk_jets, titles=["α=0 (quark)", "α=0.5", "α=1 (gluon)"]); plt.show()

<!-- cadence:task ex8-jet-recon -->
### Exercise 8 — Reconstruct gluon jets, watch the jet mass

The fair test of the VAE: **encode** real gluon jets to their latent mean and
**decode** them back. Overlay a few real jets against their reconstructions (same
axes, different colours), plot the jet-mass spine (real vs reconstructed), and
return the **1-Wasserstein distance** between the two mass distributions as
`mass_w1`. Good reconstructions sit almost on top of the real jets; the VAE's
softness shows up as a smeared high-mass tail.

In [ ]:
jvae.eval()
real_np = gluon[:512]                                   # physical real jets
real_s = torch.as_tensor(jet_std.transform(real_np)).to(device)   # standardized for the model
with torch.no_grad():
    mu_g = ...        # encode to the latent MEAN:  jvae.encoder(real_s)[:, :LATENT_DIM]
    recon_s = ...     # decode it back:             jvae.decoder(mu_g)
recon_g = jet_std.inverse_transform(recon_s.cpu().numpy())        # -> physical jets
mass_w1 = ...         # round(jet_mass_w1(real_np, recon_g), 4)

check("ex8-jet-recon", mass_w1)

In [ ]:
# real gluon jets vs your VAE reconstructions — event displays + the jet-mass spine:
plot_jet_overlay(real_np, recon_g, n=3, labels=("real gluon", "VAE recon")); plt.show()
plot_jet_mass(real_np, recon_g, labels=("real gluon", "VAE recon")); plt.show()

<!-- cadence:task ex9-jet-blur -->
### Exercise 9 — Where does the jet VAE fall short? (reflection)

Look at the jet-mass overlay you just made (real gluon vs VAE reconstruction).
VAEs tend to nail the bulk of a distribution but smear sharp features. Write 2–3
sentences and mark it done:

- Does the VAE match the real jets better in the **bulk** or in the **tail** of
  the mass distribution?
- Why would a VAE's Gaussian, averaging behaviour smear a sharp peak or tail?

In [ ]:
# Your reflection (2-3 sentences), replace the string below with your 2-3 sentences, then run the cell:
#   * bulk or tail — where does the reconstruction match worst?
#   * why does a VAE smear sharp features?

check("ex9-jet-blur", "...your 2-3 sentences here...")

## Recap

- **PPCA** is the linear/Gaussian ancestor of the VAE: same latent-variable
  skeleton, closed-form fit, but a *flat* subspace — so its samples stay blurry.
- A VAE gives a **smooth latent space** you can sample and interpolate in — the
  same skeleton made *nonlinear*.
- **Latent arithmetic** works because the space is organised by meaningful
  directions (cat → dog; quark → gluon).
- On jets, a **permutation-invariant GNN encoder** respects the point-cloud
  structure — and the **jet-mass spine** shows the classic VAE weakness in the
  tail, which we'll compare against a GAN (02) and diffusion (03, 05).